# 五会话标签的有效样本与保存Ridge设计矩阵

只读Qlib/Recorder：重新装配同一成熟训练特征，使用保存的SimpleImputer/StandardScaler与SOXX候选Ridge，核输入顺序、预测身份、按交易日权重的Gram秩及五会话不重叠来源上界。对已知2022Q2首分钟只报告保存线性模型正负贡献抵消；不读新的经济目标、不拟合/改alpha。

In [1]:
import os,json,hashlib,warnings
from pathlib import Path
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';os.environ['MLFLOW_DISABLE_AGENT_HINT']='1'
warnings.filterwarnings('ignore',message="The 'generic' unit for NumPy timedelta")
import numpy as np
import pandas as pd
import qlib
from qlib.data import D
from src.core.recorder_sources import load_verified_recorder_objects
from src.analysis.stage1_fixed_share_drawdown_information import prepare_feature_panel
from src.analysis.stage1_peer_downshock_information import attach_peer_downshock
root=Path.cwd();card=root/'research/stage1_v5_effective_sample_design_audit_20261030';plan=json.loads((card/'plan.json').read_text());sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
info=root/plan['information_card'];assert sha(info/'summary.json')==plan['information_summary_sha256'] and sha(info/'verification_manifest.json')==plan['information_manifest_sha256'] and json.loads((info/'nas_binding.json').read_text())['evidence']['sha256']==plan['information_nas_archive_sha256']
for role in ('label_source','parent','frame'):
 s=plan[role];source=root/s['card'];assert sha(source/'summary.json')==s['summary_sha256'] and sha(source/'verification_manifest.json')==s['manifest_sha256'] and json.loads((source/'nas_binding.json').read_text())['evidence']['sha256']==s['nas_archive_sha256']
assert sha(root/plan['core_provider_uri']/'qlib_store_manifest.json')==plan['core_provider_manifest_sha256'] and sha(root/plan['peer_provider_uri']/'qlib_store_manifest.json')==plan['peer_provider_manifest_sha256'] and sha(root/plan['support_card']/'perday_support.csv')==plan['support_perday_sha256']
source_plan=json.loads((info/'plan.json').read_text())
qlib.init(provider_uri=str(root/plan['peer_provider_uri']),region='us',kernels=1)
peer=D.features(['QQQ','SMH','NVDA','AMD'],['$open','$close','$price_valid'],start_time='2018-10-15 09:35:00',end_time='2022-12-22 16:00:00',freq='5min')
qlib.init(provider_uri=str(root/plan['core_provider_uri']),region='us',kernels=1)
def read(role,name):
 s=plan[role];v,_=load_verified_recorder_objects(root/s['card'],expected_recorder_id=s['recorder_id'],artifacts=[name],expected_manifest_sha256=s['manifest_sha256']);return v[name]
info_spec={'card':plan['information_card'],'recorder_id':plan['information_recorder_id'],'manifest_sha256':plan['information_manifest_sha256']}
model=read('information_card_dummy','x') if False else None
v,_=load_verified_recorder_objects(info,expected_recorder_id=plan['information_recorder_id'],artifacts=['models/candidate_soxx','analysis/predictions'],expected_manifest_sha256=plan['information_manifest_sha256'])
model=v['models/candidate_soxx'];saved_pred=v['analysis/predictions'];labels=read('label_source','analysis/labels');book=read('parent','analysis/native/dated_f696_0');frame=read('frame','inputs/covered_frame')
core=D.features(['SOXL','SOXX'],['$close','$raw_volume','$price_valid'],start_time='2018-10-15 09:31:00',end_time='2022-12-22 16:00:00',freq='1min')
core_plan=dict(source_plan);core_plan['state_features']=[x for x in source_plan['state_features'] if x!='own_drawdown_change'];core_plan['ordinary_minute_features']=source_plan['ordinary_minute_features'][:6];core_plan['candidate_only_feature']='own_drawdown_change'
panel,_=prepare_feature_panel(labels,frame,book,core,core_plan);panel,_=attach_peer_downshock(panel,peer)
print('verified saved Qlib sources and feature panel',len(panel))

[55746:MainThread](2026-10-30 17:54:22,264) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[55746:MainThread](2026-10-30 17:54:22,661) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[55746:MainThread](2026-10-30 17:54:22,661) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/minute_information_pit_allocator_20261012/peer_5min')}


[55746:MainThread](2026-10-30 17:54:22,976) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[55746:MainThread](2026-10-30 17:54:22,977) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[55746:MainThread](2026-10-30 17:54:22,977) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/stage1_event_guard_fullminute_20261028')}


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


verified saved Qlib sources and feature panel 385932


In [2]:
first_eval=panel.loc[panel.trade_date.dt.year.eq(2021),'decision_at'].min()
train=panel.loc[panel.trade_date.dt.year.le(2020)&panel.label_available_at.lt(first_eval)&panel.status.eq('paired_funded')].copy()
evaluation=panel.loc[panel.trade_date.dt.year.isin([2021,2022])&panel.trade_date.le(pd.Timestamp('2022-12-16'))].copy()
assert len(train)==200403 and train.trade_date.nunique()==549 and len(evaluation)==len(saved_pred)==180938
names=[*source_plan['base39_features'],*source_plan['state_features'],*source_plan['ordinary_minute_features'],source_plan['candidate_only_feature']]
Xtrain=pd.concat({'feature':train[names]},axis=1);Xeval=pd.concat({'feature':evaluation[names]},axis=1)
scale=model.named_steps['standardscaler'];imputer=model.named_steps['simpleimputer'];ridge=model.named_steps['ridge']
z=scale.transform(imputer.transform(Xtrain));z_eval=scale.transform(imputer.transform(Xeval));pred=ridge.predict(z_eval)
assert evaluation.trade_date.reset_index(drop=True).equals(pd.to_datetime(saved_pred.trade_date).reset_index(drop=True))
max_prediction_error=float(np.max(np.abs(pred-saved_pred.candidate_soxx.to_numpy(float))));assert max_prediction_error<1e-12
counts=train.trade_date.value_counts();weights=1./train.trade_date.map(counts).to_numpy(float)
gram=z.T@(z*weights[:,None]);eigen=np.linalg.eigvalsh((gram+gram.T)/2);eigen=np.clip(eigen,0,None)
rank=int(np.sum(eigen>eigen.max()*1e-10));participation=float(eigen.sum()**2/np.square(eigen).sum());ridge_condition=float((eigen.max()+1)/(eigen.min()+1))
missing_names=list(imputer.get_feature_names_out(names));dimension=len(missing_names)
weights_sum=float(weights.sum());assert abs(weights_sum-549)<1e-9
raw_f696_sum_error=float(np.max(np.abs(train.soxl_weight+train.soxx_weight+train.cash_weight-1)))
raw_decision_sum_error=float(np.max(np.abs(train.decision_soxl_weight+train.decision_soxx_weight+train.decision_cash_weight-1)))
support=pd.read_csv(root/plan['support_card']/'perday_support.csv',parse_dates=['trade_date']);selected=[];next_allowed=0
for i,row in support.iterrows():
 if i>=next_allowed and row.paired_funded>0:selected.append((i,int(row.year)));next_allowed=i+5
byyear={str(y):int(sum(yy==y for _,yy in selected)) for y in sorted(support.year.unique())}
first=(evaluation.bar_end_at.dt.hour.eq(9)&evaluation.bar_end_at.dt.minute.eq(48)).to_numpy();q2=evaluation.trade_date.dt.to_period('Q').astype(str).eq(plan['weak_period']).to_numpy()&first
contrib=z_eval[q2]*ridge.coef_;mean_contrib=contrib.mean(axis=0);net=float(mean_contrib.sum());absolute=float(np.abs(mean_contrib).sum());cancel_ratio=float(absolute/max(abs(net),1e-12))
shift=pd.DataFrame({'feature':missing_names,'mean_2022Q2_firstbar_contribution':mean_contrib}).sort_values('mean_2022Q2_firstbar_contribution',key=np.abs,ascending=False)
shift.to_csv(card/'firstbar_coefficient_contributions.csv',index=False)
audit={'experiment':plan['experiment'],'plan_sha256':sha(card/'plan.json'),'source_recorder_id':plan['information_recorder_id'],'model_feature_columns':len(names),'transformed_dimension':dimension,'train_minute_rows':len(train),'train_distinct_days':int(train.trade_date.nunique()),'train_daily_weight_sum':weights_sum,'eval_minute_rows':len(evaluation),'eval_distinct_days':int(evaluation.trade_date.nunique()),'five_session_nonoverlap_source_upper_bound_total':len(selected),'five_session_nonoverlap_source_upper_bound_by_year':byyear,'train_2018_2020_nonoverlap_upper_bound':sum(byyear[str(y)] for y in (2018,2019,2020)),'eval_2021_2022_nonoverlap_upper_bound':sum(byyear[str(y)] for y in (2021,2022)),'gram_numeric_rank_at_maxeig_1e_minus10':rank,'gram_participation_rank':participation,'gram_ridge_alpha1_condition':ridge_condition,'smallest_eigen':float(eigen.min()),'largest_eigen':float(eigen.max()),'f696_simplex_max_error':raw_f696_sum_error,'decision_simplex_max_error':raw_decision_sum_error,'saved_prediction_max_error':max_prediction_error,'2022Q2_firstbar_mean_score_contribution_net':net,'2022Q2_firstbar_mean_score_contribution_absolute_sum':absolute,'2022Q2_firstbar_cancellation_ratio':cancel_ratio,'top_abs_2022Q2_firstbar_contributions':shift.head(12).to_dict('records'),'new_model_fits':0,'new_candidate_accounts':0,'historical_research_passed':False,'decision':'saved_high_dimensional_overlapping_design_diagnostic_no_policy'}
(card/'audit.json').write_text(json.dumps(audit,ensure_ascii=False,indent=2)+'\n')
print(json.dumps({'dimension':dimension,'train_days':audit['train_distinct_days'],'nonoverlap_train':audit['train_2018_2020_nonoverlap_upper_bound'],'nonoverlap_eval':audit['eval_2021_2022_nonoverlap_upper_bound'],'rank':rank,'participation':participation,'condition':ridge_condition,'cancel_ratio':cancel_ratio,'top':audit['top_abs_2022Q2_firstbar_contributions'][:6]},ensure_ascii=False,indent=2))

{
  "dimension": 62,
  "train_days": 549,
  "nonoverlap_train": 112,
  "nonoverlap_eval": 99,
  "rank": 58,
  "participation": 9.54635770440889,
  "condition": 8885.236691018457,
  "cancel_ratio": 7.2584251945237925,
  "top": [
    {
      "feature": "dist_high_252",
      "mean_2022Q2_firstbar_contribution": 0.011407250488711078
    },
    {
      "feature": "range_pos_252",
      "mean_2022Q2_firstbar_contribution": -0.010031206778852423
    },
    {
      "feature": "rv20_expansion",
      "mean_2022Q2_firstbar_contribution": -0.004212944585927755
    },
    {
      "feature": "rv20",
      "mean_2022Q2_firstbar_contribution": 0.003748878725543692
    },
    {
      "feature": "soxx_trend60",
      "mean_2022Q2_firstbar_contribution": 0.0035461918255711715
    },
    {
      "feature": "soxx_vol20",
      "mean_2022Q2_firstbar_contribution": 0.003163899192600207
    }
  ]
}


# v5 保存模型有效样本与设计审计

**SOXX 判定：历史研究未通过。**本卡没有新拟合、整账户或0/2/5bp候选回测；沿用已关闭的条件跨资产5分钟信息模型与已验证Qlib Recorder，原固定信息门仍失败。2022Q2的62日首分钟候选均买SOXX，预测SOXX相对HOLD约+91.748bp，实际五会话整股Tiered费后约−20.859bp；匹配控制62日也均首分钟买SOXX、候选仅1日改为另一资产。损失主要是后续股价路径，首单费均约0.020bp，不能归因为新增条件因子一项。

本次只读复算保存的SOXX候选Ridge设计：2018–2020训练表200403分钟行，只有549个不同日；每个交易日总权重1。五会话不重叠来源起点的**上界**是112，2021/22评价180938行/494日的对应上界99。重叠路径和共同日状态使20万行不能视为20万个独立经济样本；上界不是精确有效样本量。

62列标准化输入的加权Gram数值秩58，谱参与秩9.546；f696提案权重与决策权重各严格合为1，确有线性依赖。alpha=1的正则化条件数约8885。保存预测复算最大误差0。2022Q2首分钟分数的特征平均贡献净值+83.106bp，绝对贡献和603.218bp，抵消比7.26；`dist_high_252`约+114.073bp，`range_pos_252`约−100.312bp。它说明高维互相抵消和跨年分数漂移值得控制，**不能证明**秩缺陷单独造成真实亏损，也不能根据2022Q2事后调正则、选特征或时段。

研究方法修正：任何新接收器须先提供未关闭的决策时经济机制、较低预定自由度、按日分组并隔离五会话重叠的训练/校准方案，以及从分数到实际整股订单的正向传导门。原模型与同源字段加法保持关闭；只读有限候选来源审计后若仍无独立机制，记录`needs_distinct_decision_time_mechanism`。本卡无新增完整账户，不能判断新架构是否跑赢SOXX。来源和原始计算见`plan.json`、`audit.json`、`firstbar_coefficient_contributions.csv`；首次JSON序列化失败及修正保留在`attempt1_serialization_failure.json`和`plan_v1_before_json_int_fix.json`。
